In [ ]:
#Install needed libraries and dependencies
!pip uninstall -y vllm transformers tokenizers
!pip install vllm==0.6.4.post1 transformers==4.46.3 tokenizers==0.20.3 accelerate pandas

In [ ]:
#Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
#Tier 2
import os
import gc
import glob
import torch
import pandas as pd
from transformers import AutoTokenizer
from vllm import LLM, SamplingParams

gc.collect()
torch.cuda.empty_cache()

MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct-AWQ" #Rationale Generation Model
DATASET_PATH = "PATH_WHERE_TIER_1_OUTPUT_SAVED"
SAVE_DIR = "PATH_WHERE_OUTPUT_FILE_IS_STORED"
os.makedirs(SAVE_DIR, exist_ok=True)
CHUNK_SIZE = 100
MASTER_FILE = os.path.join(SAVE_DIR, "RATIONALE_FILE_NAME.csv")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

#vLLM initialize
llm = LLM(
    model=MODEL_NAME,
    quantization="awq",
    dtype="half",
    gpu_memory_utilization=0.90,
    max_model_len=4096,
    max_num_seqs=8,
    trust_remote_code=True,
    enforce_eager=True,
    disable_custom_all_reduce=True
)

if os.path.exists(MASTER_FILE):
    print("Resuming from existing Master Progress file...")
    df = pd.read_csv(MASTER_FILE)
else:
    print("Starting fresh from original dataset...")
    df = pd.read_csv(DATASET_PATH)

# Ensure English column exists
target_col = 'question_en'
if target_col not in df.columns:
    raise ValueError(f"Column '{target_col}' not found. Check your CSV header!")

questions = df[target_col].tolist()

#Prompt for Rational Generator Model
prompts = []
for q in questions:
    messages = [
        {"role": "system", "content": "You are a professional educational and guidance expert. Provide a concise, formal rationale explaining the concepts behind the user's question. Do NOT use inner monologue. Do NOT state the final correct answer or mention any options. Provide only the objective explanation."},
        {"role": "user", "content": f"Question: {q}"}
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    prompts.append(text)

#Temperature-based rationales
sampling_configs = {
    "Logical": SamplingParams(temperature=0.2, top_p=0.95, max_tokens=350),
    "Balanced": SamplingParams(temperature=0.45, top_p=0.95, max_tokens=350),
    "Creative": SamplingParams(temperature=0.7, top_p=0.90, max_tokens=400)
}

for mode, params in sampling_configs.items():
    col_name = f'rationale_{mode}'

    if col_name in df.columns and df[col_name].notna().sum() == len(df):
        print(f"Mode {mode} is already complete. Skipping...")
        continue

    print(f"\n>>> Processing Mode: {mode}")

    if col_name not in df.columns:
        df[col_name] = ""

    start_idx = df[df[col_name].fillna("") == ""].index.min()
    if pd.isna(start_idx):
        continue

    print(f"Resuming {mode} from row {start_idx}...")

    for i in range(start_idx, len(prompts), CHUNK_SIZE):
        end_idx = min(i + CHUNK_SIZE, len(prompts))
        chunk_prompts = prompts[i : end_idx]

        outputs = llm.generate(chunk_prompts, params)
        chunk_results = [output.outputs[0].text.strip() for output in outputs]

        df.loc[i : end_idx-1, col_name] = chunk_results

        df.to_csv(MASTER_FILE, index=False, encoding='utf-8-sig')
        print(f"Saved checkpoint: {mode} rows {i} to {end_idx}")

print("\nAll tasks finished. Final file is located at:", MASTER_FILE)